# Pipeline de Transformação: estoque

In [ ]:
import sys
from pathlib import Path

# Adiciona o diretório raiz do projeto ao sys.path para importações locais
sys.path.append(str(Path.cwd().parent.parent))

from pyspark.sql import functions as F
from src.modules.spark_session import get_spark_session, close_spark_session
import src.modules.transform_utils as transform
import src.modules.utils as utils

In [ ]:
# Inicializa a SparkSession conectada ao cluster do container
spark = get_spark_session("TransformEstoque")

# Leitura dos dados da camada Bronze

In [ ]:
# Caminho da tabela Bronze no MinIO
bronze_path = "s3a://bronze/estoque"

# Lê os dados da Bronze
try:
    df_estoque = spark.read.parquet(bronze_path).withColumnRenamed("data_carga", "data_carga_bronze")
except Exception as e:
    print(f"Erro ao ler a camada Bronze: {e}")

print(f"Total de registros na camada Bronze: {df_estoque.count()}")
df_estoque.printSchema()
df_estoque.limit(5).toPandas()

# Aplica TRIM nas colunas de texto

In [ ]:
text_cols = ["produto", "categoria", "marca", "fornecedor", "centro_distribuicao"]
df_estoque = transform.trim_columns(df_estoque, text_cols)

df_estoque.select(*text_cols).distinct().limit(5).toPandas()

# Arredonda a coluna de valor para 2 casas decimais

In [ ]:
value_cols = ["custo_unitario"]
df_estoque = transform.round_values(df_estoque, value_cols, decimals=2)

df_estoque.select(*value_cols).limit(5).toPandas()

# Adiciona a data de carga da transformação

In [ ]:
# Adiciona a data de carga do processamento da Silver
df_estoque = df_estoque.withColumn("data_carga", F.to_date(F.lit(utils.get_current_date_str())))

df_estoque.select("produto_id", "data_carga").limit(5).toPandas()

In [ ]:
# Finaliza a sessão do Spark
close_spark_session(spark)